# Time-Series Forecasting with ARIMA
**Task 05 – RobTech Academy**

This notebook analyzes daily demand/sales data, checks stationarity with the Augmented Dickey-Fuller (ADF) test, decomposes the time series, trains an ARIMA model, evaluates it using MAPE and RMSE, and forecasts the next 30 days.

**Before running:** Keep the downloaded `Daily demand time series.csv` file in the same folder as this notebook. If your CSV has different column names, the notebook will try to identify the date and demand columns automatically; you can also set them manually in the data-loading cell.


## 1. Install and import required libraries
Run the installation command once if any library is missing.

In [ ]:
# If needed, uncomment and run this line:
# %pip install pandas numpy matplotlib statsmodels scikit-learn

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)


## 2. Load the dataset
Place `Daily demand time series.csv` beside this notebook. If your file has another name, change `file_path` below.

In [ ]:
file_path = "Daily demand time series.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumn names:", list(df.columns))


## 3. Prepare the time-series data
The code attempts to identify date and demand columns. If it selects the wrong columns, replace `date_col` and `value_col` with the correct column names shown above.

In [ ]:
# Automatic column detection (edit these two values manually if necessary)
date_candidates = [c for c in df.columns if any(k in c.lower() for k in ["date", "time", "day"])]
value_candidates = [c for c in df.columns if any(k in c.lower() for k in ["demand", "sales", "value", "quantity", "y"])]

date_col = date_candidates[0] if date_candidates else df.columns[0]
value_col = value_candidates[0] if value_candidates else df.columns[1]

print("Date column:", date_col)
print("Demand/Sales column:", value_col)

data = df[[date_col, value_col]].copy()
data.columns = ["Date", "Demand"]
data["Date"] = pd.to_datetime(data["Date"], errors="coerce", dayfirst=True)
data["Demand"] = pd.to_numeric(data["Demand"].astype(str).str.replace(",", ""), errors="coerce")
data = data.dropna().sort_values("Date")
data = data.groupby("Date", as_index=True)["Demand"].mean().to_frame()

# Convert to daily frequency and fill missing days using time interpolation
data = data.asfreq("D")
data["Demand"] = data["Demand"].interpolate(method="time").ffill().bfill()

print("Prepared data range:", data.index.min(), "to", data.index.max())
display(data.head())
print("Missing values after preparation:", data["Demand"].isna().sum())


## 4. Visualize daily demand

In [ ]:
plt.figure()
plt.plot(data.index, data["Demand"], label="Daily demand")
plt.title("Daily Demand Over Time")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.grid(True)
plt.show()


## 5. Time-series decomposition
This separates the series into trend, seasonal, and residual components. A weekly period (7 days) is used for daily data.

In [ ]:
if len(data) >= 14:
    decomposition = seasonal_decompose(data["Demand"], model="additive", period=7, extrapolate_trend="freq")
    decomposition.plot()
    plt.suptitle("Time-Series Decomposition (7-day seasonality)", y=1.02)
    plt.show()
else:
    print("At least 14 daily observations are needed for weekly decomposition.")


## 6. Augmented Dickey-Fuller (ADF) stationarity test
A p-value below 0.05 is commonly interpreted as evidence that the series is stationary. ARIMA with `d=1` is used below as a simple baseline.

In [ ]:
adf_result = adfuller(data["Demand"].dropna())
print("ADF Statistic:", round(adf_result[0], 4))
print("p-value:", round(adf_result[1], 6))
print("Critical values:")
for key, value in adf_result[4].items():
    print(f"  {key}: {value:.4f}")

if adf_result[1] < 0.05:
    print("\\nConclusion: The series appears stationary (at 5% significance).")
else:
    print("\\nConclusion: The series appears non-stationary (at 5% significance).")


## 7. Train-test split
The last 20% of observations are held out for evaluation. The model is fitted only on the training portion.

In [ ]:
if len(data) < 20:
    raise ValueError("The dataset is too short for a reliable train/test evaluation. Please use at least 20 daily observations.")

split_index = int(len(data) * 0.8)
train = data["Demand"].iloc[:split_index]
test = data["Demand"].iloc[split_index:]

print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Test period:", test.index.min(), "to", test.index.max())


## 8. Train the ARIMA model
This notebook uses ARIMA(1,1,1) as a clear baseline model. For a formal project, compare multiple parameter combinations and select the best-performing model.

In [ ]:
model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()

print(model_fit.summary())


## 9. Predict the test period and evaluate accuracy
MAPE is reported as a percentage. RMSE is in the same units as the demand data.

In [ ]:
test_predictions = model_fit.forecast(steps=len(test))
test_predictions = pd.Series(test_predictions.values, index=test.index)

# Avoid undefined MAPE if actual values contain zeros
nonzero_mask = test != 0
if nonzero_mask.any():
    mape = mean_absolute_percentage_error(test[nonzero_mask], test_predictions[nonzero_mask]) * 100
else:
    mape = np.nan

rmse = np.sqrt(mean_squared_error(test, test_predictions))

print(f"MAPE: {mape:.2f}%")
print(f"RMSE: {rmse:.2f}")


In [ ]:
plt.figure()
plt.plot(train.index, train, label="Training data")
plt.plot(test.index, test, label="Actual test data")
plt.plot(test_predictions.index, test_predictions, label="Predicted test data")
plt.title("Actual vs Predicted Demand (Test Set)")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.grid(True)
plt.show()


## 10. Forecast the next 30 days
The final ARIMA model is fitted on all available observations before generating the 30-day forecast.

In [ ]:
final_model = ARIMA(data["Demand"], order=(1, 1, 1))
final_model_fit = final_model.fit()

forecast_steps = 30
forecast_values = final_model_fit.forecast(steps=forecast_steps)

last_date = data.index.max()
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_steps, freq="D")
forecast = pd.Series(np.asarray(forecast_values), index=forecast_dates, name="Forecast")

forecast_table = forecast.to_frame()
display(forecast_table.head(10))


In [ ]:
plt.figure()
plt.plot(data.index, data["Demand"], label="Historical demand")
plt.plot(forecast.index, forecast, label="30-day forecast")
plt.title("30-Day Demand Forecast using ARIMA")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.grid(True)
plt.show()


## 11. Save forecast results
This creates a CSV file containing the next 30 days of forecast values.

In [ ]:
forecast.to_frame().to_csv("30_day_demand_forecast.csv", index_label="Date")
print("Saved: 30_day_demand_forecast.csv")


## 12. Conclusion

- The daily demand series was loaded, cleaned, and visualized.
- Trend, seasonal, and residual components were examined using decomposition.
- The ADF test was used to assess stationarity.
- An ARIMA(1,1,1) baseline model was trained and evaluated.
- MAPE and RMSE were calculated on a held-out test set.
- A 30-day demand forecast was generated and saved as a CSV file.

**Submission:** Save this notebook after running all cells. Submit `Time_Series_Forecasting.ipynb` as the Jupyter Notebook proof requested by the task. If the academy accepts supporting files, include the dataset and `30_day_demand_forecast.csv` as well.

**Important:** Your actual MAPE, RMSE, graphs, and forecast will appear only after you run the notebook with the academy's downloaded dataset. Do not submit blank/unexecuted result cells if the academy expects outputs.
